In [4]:
%pip install pythae

  Using cached pythae-0.1.2-py3-none-any.whl.metadata (52 kB)
  Using cached cloudpickle-3.1.2-py3-none-any.whl.metadata (7.1 kB)
  Using cached imageio-2.37.3-py3-none-any.whl.metadata (9.7 kB)
  Using cached dataclasses-0.6-py3-none-any.whl.metadata (3.0 kB)
  Using cached annotated_types-0.7.0-py3-none-any.whl.metadata (15 kB)
  Using cached typing_inspection-0.4.2-py3-none-any.whl.metadata (2.6 kB)
  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
Using cached pythae-0.1.2-py3-none-any.whl (235 kB)
Using cached cloudpickle-3.1.2-py3-none-any.whl (22 kB)
Using cached dataclasses-0.6-py3-none-any.whl (14 kB)
   ---------------------------------------- 0.0/2.1 MB ? eta -:--:--
   ---------------------------------------- 2.1/2.1 MB 14.6 MB/s  0:00:00
Using cached annotated_types-0.7.0-py3-none-any.whl (13 kB)
   ---------------------------------------- 0.0/37.3 MB ? eta -:--:--
   --- ------------------------------------ 2.9/37.3 MB 14.7 MB/s eta 0:00:03
   ----- ---


[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import os

class ImageFolderFlat(Dataset):
    def __init__(self, root, transform=None):
        self.paths = [
            os.path.join(root, f) for f in os.listdir(root)
            if f.lower().endswith((".jpg", ".jpeg", ".png", ".bmp", ".webp"))
        ]
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        img = Image.open(self.paths[idx]).convert("RGB")  # use "L" for grayscale
        if self.transform:
            img = self.transform(img)
        return img, 0  # dummy label so it works with the training loop unchanged

transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor(),
    transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5]),
])

dataset = ImageFolderFlat("./data/pokemon_data/train", transform=transform)
loader  = DataLoader(dataset, batch_size=16, shuffle=True, num_workers=4, pin_memory=True)

In [ ]:
from pythae.models import AE, AEConfig
from pythae.trainers import BaseTrainer, BaseTrainerConfig

model_config = AEConfig(input_dim=(3, 64, 64), latent_dim=16)
model = AE(model_config=model_config)

trainer_config = BaseTrainerConfig(num_epochs=10, learning_rate=1e-3, batch_size=16)
trainer = BaseTrainer(model=model, train_dataset=loader.dataset, training_config=trainer_config)
trainer.train()

# Extract embeddings
encoded = model.encoder(image_tensor)  # → (N, 16)

! No eval dataset provided ! -> keeping best model on train.



ModelError: Error when calling forward method from model. Potential issues: 
 - Wrong model architecture -> check encoder, decoder and metric architecture if you provide yours 
 - The data input dimension provided is wrong -> when no encoder, decoder or metric provided, a network is built automatically but requires the shape of the flatten input data.
Exception raised: <class 'TypeError'> with message: list indices must be integers or slices, not str

In [3]:
class ImageFolderFlat(Dataset):
    def __init__(self, root, transform=None):
        self.paths = [
            os.path.join(root, f) for f in os.listdir(root)
            if f.lower().endswith((".jpg", ".jpeg", ".png", ".bmp", ".webp"))
        ]
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        img = Image.open(self.paths[idx]).convert("RGB")  # use "L" for grayscale
        if self.transform:
            img = self.transform(img)
        return img, 0  # dummy label so it works with the training loop unchanged

NameError: name 'Dataset' is not defined

In [43]:
import torch
from torch import optim, nn

def train(model, loader, epochs=30, lr=1e-3, device="cpu"):
    model.to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    criterion = nn.MSELoss()

    for epoch in range(epochs):
        model.train()
        total = 0
        for imgs, _ in loader:
            imgs = imgs.to(device)
            optimizer.zero_grad()
            recon = model(imgs)
            loss = recon.loss
            loss.backward()
            optimizer.step()
            total += loss.item()
        if (epoch + 1) % 5 == 0:
            print(f"Epoch {epoch+1}/{epochs}  Loss: {total/len(loader):.4f}")


# ── Embeddings ───────────────────────────────────────────────────────────────

def get_embeddings(model, loader, device="cpu"):
    model.eval()
    all_z = []
    with torch.no_grad():
        for imgs, _ in loader:
            z = model.encoder(imgs.to(device)).embedding
            print(z)
            all_z.append(z)
    return torch.cat(all_z).numpy()


In [44]:
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split
import torch

from pythae.models import AE, AEConfig
from pythae.trainers import BaseTrainer, BaseTrainerConfig

# 1. Dataset & loader
transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor(),
    transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5]),
])

dataset = ImageFolderFlat(root="./Images/0/", transform=transform)

n_val = int(0.1 * len(dataset))
train_ds, val_ds = random_split(dataset, [len(dataset) - n_val, n_val])

train_loader = DataLoader(train_ds, batch_size=16, shuffle=True)

# 2. Model
device = "cuda" if torch.cuda.is_available() else "cpu"

model_config = AEConfig(input_dim=(3, 64, 64), latent_dim=16)
model = AE(model_config=model_config)

# 3. Train — pass train_loader directly
train(model, train_loader, epochs=10, device=device)

# 4. Extract embeddings — pass whichever loader you need
#train_embeddings = get_embeddings(model, train_loader, device=device)  # (N_train, 16)


KeyboardInterrupt: 

In [ ]:
transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor(),
    transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5]),
])

dataset = datasets.ImageFolder(root="./data/pokemon_data/", transform=transform)


In [45]:
train_embeddings = get_embeddings(model, train_loader, device=device)

tensor([[ 2.8722e+00,  3.4979e+00,  3.7046e+00,  9.9202e-01, -5.6220e+00,
          7.4546e+00,  7.9139e+00, -4.5443e+00, -2.9486e+00, -2.1277e+00,
         -2.1579e+00, -6.6108e-01, -1.8726e+00, -5.6887e+00,  2.2789e+00,
         -6.1984e+00],
        [ 4.6239e+00,  2.0765e+00,  4.8150e+00,  2.5202e+00, -4.5609e-01,
          8.8343e+00,  6.7554e+00, -6.1026e+00, -8.9657e-01, -4.3198e+00,
         -4.6454e+00,  2.3081e+00,  2.1011e-01, -4.0124e+00,  2.1069e+00,
         -1.3795e+00],
        [ 1.4880e-01,  5.6604e+00,  3.7544e+00, -1.1650e-01, -1.0892e+01,
          6.1549e+00,  9.0083e+00, -1.7976e+00, -5.0978e+00,  7.4834e-01,
          2.0708e+00, -4.2571e+00, -5.1841e+00, -7.7376e+00,  3.4024e+00,
         -1.1338e+01],
        [-4.3127e+01,  4.6930e+01,  1.2936e+01, -9.4520e+00, -8.7461e+01,
         -8.4323e+00,  2.4185e+01,  4.4328e+01, -3.6040e+01,  4.0482e+01,
          6.6080e+01, -4.6478e+01, -5.5322e+01, -3.9288e+01,  2.3297e+01,
         -8.2732e+01],
        [ 3.3328e+00

In [46]:
train_embeddings.shape

(223, 16)